XGBoost In-Class Example

In [1]:
import pandas as pd
import numpy as np
import sweetviz as sv
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from xgboost import XGBClassifier

c:\Users\prchandr\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
pip install XGBoost

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
default_df = pd.read_csv(r'c:\Users\prchandr\Downloads\creditcard.csv')
default_df.head()

,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,400000,1,1,2,32,0,0,0,0,0,...,48272,49478,51242,3028,3023,3000,3000,3000,38662,0
1,120000,2,2,2,30,-1,-1,-1,-1,-1,...,1964,1883,1538,3230,3011,1964,1883,1538,1911,0
2,270000,2,2,2,32,0,0,0,0,0,...,94856,86461,83650,1808,69563,2891,2689,3012,2771,0
3,280000,2,2,1,27,0,0,0,0,0,...,257689,193231,191143,11052,9563,15017,5374,5420,6021,0
4,30000,2,1,2,27,0,0,-1,0,0,...,1814,0,0,1000,664,1500,0,0,0,0


In [4]:
report = sv.analyze(default_df)
report.show_html('sweet_report.html')

Done! Use 'show' commands to display/save.   |██████████| [100%]   00:01 -> (00:00 left)


Report sweet_report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


In [5]:
default = default_df.copy()

Random Forest

In [6]:
X = default.drop('default payment next month', axis=1)
y = default['default payment next month']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)
print("Random Forest Classification Report:")
print(classification_report(y_test, y_pred_rf))
print("Random Forest Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))
print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))

Random Forest Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.94      0.89      3720
           1       0.64      0.38      0.47      1080

    accuracy                           0.81      4800
   macro avg       0.74      0.66      0.68      4800
weighted avg       0.79      0.81      0.79      4800

Random Forest Confusion Matrix:
[[3490  230]
 [ 675  405]]
Random Forest Accuracy: 0.8114583333333333


XGBoost Model

In [9]:
# clean columns
default_df.columns = default_df.columns.str.strip()

# Target column
y = default_df['default payment next month']
X = default_df.drop(columns=['default payment next month'])

# Drop ID if present
if 'ID' in X.columns:
    X = X.drop(columns=['ID'])

# Train-test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Build and train model
xgb_model = XGBClassifier(
    random_state=42,
    eval_metric='logloss'
)

xgb_model.fit(X_train, y_train)

# Predict
y_pred_xgb = xgb_model.predict(X_test)

# Evaluate
print("XGBoost Classification Report:")
print(classification_report(y_test, y_pred_xgb))

print("XGBoost Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_xgb))

XGBoost Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.93      0.88      3738
           1       0.61      0.38      0.47      1062

    accuracy                           0.81      4800
   macro avg       0.73      0.66      0.68      4800
weighted avg       0.79      0.81      0.79      4800

XGBoost Confusion Matrix:
[[3483  255]
 [ 656  406]]


Hypermeters for XGBoost

In [ ]:
# hyperparameters for xgboost using 4 parameters
param_grid_xgb = {
    'n_estimators': [100, 200],
    'max_depth': [3, 6],
    'learning_rate': [0.01, 0.1],
    'subsample': [0.8, 1.0]
}
# Create a fresh estimator for GridSearchCV (avoids cloning issues with a fitted estimator)
grid_xgb = GridSearchCV(
    estimator=XGBClassifier(random_state=42, eval_metric='logloss'),
    param_grid=param_grid_xgb,
    cv=3,
    n_jobs=-1,
    verbose=2
)
grid_xgb.fit(X_train.astype("float64").to_numpy(), y_train.astype("int64").to_numpy())
print("Best parameters for XGBoost:", grid_xgb.best_params_)
best_xgb = grid_xgb.best_estimator_
y_pred_xgb_tuned = best_xgb.predict(X_test.astype("float64").to_numpy())
print("Tuned XGBoost Classification Report:")
print(classification_report(y_test.astype("int64").to_numpy(), y_pred_xgb_tuned))
print("Tuned XGBoost Confusion Matrix:")
print(confusion_matrix(y_test.astype("int64").to_numpy(), y_pred_xgb_tuned))
print("Tuned XGBoost Accuracy:", accuracy_score(y_test.astype("int64").to_numpy(), y_pred_xgb_tuned))

Fitting 3 folds for each of 16 candidates, totalling 48 fits
Best parameters for XGBoost: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100, 'subsample': 0.8}
Tuned XGBoost Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.95      0.89      3738
           1       0.67      0.37      0.48      1062

    accuracy                           0.82      4800
   macro avg       0.75      0.66      0.68      4800
weighted avg       0.80      0.82      0.80      4800

Tuned XGBoost Confusion Matrix:
[[3540  198]
 [ 667  395]]
Tuned XGBoost Accuracy: 0.8197916666666667
